<a href="https://colab.research.google.com/github/sw030701-ai/motor-control-optimization/blob/main/experiments/03_pid_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 · Constrained PID Optimization Experiment
### Manual Baseline PID vs Constrained Classical Optimization

---

### Objective

이 notebook의 목적은 앞에서 만든 **Manual Baseline PID**를 benchmark로 두고, 동일한 nominal motor / reference / simulation condition에서 v1 constraints를 만족하는 classical PID gains를 찾는 것이다.

```text
Manual Baseline PID
      ↓
Same plant + same reference + same simulation settings
      ↓
Constrained Random Search + Constrained Bayesian Optimization
      ↓
Select best feasible method by J
      ↓
Manual Baseline vs Constrained Classical comparison
```

핵심 원칙은 **Feasibility first, optimization second**이다. Cost가 낮아도 v1 constraints를 만족하지 못하면 최종 PID로 선택하지 않는다.

In [1]:
import os, sys, json, platform, subprocess, math, warnings
from pathlib import Path


def _in_colab():
    return "google.colab" in sys.modules


def _find_root(start: Path) -> Path:
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "src").exists() and (cand / "docs").exists():
            return cand
    return p


REPO_URL = "https://github.com/sw030701-ai/motor-control-optimization.git"

if _in_colab():
    root = Path("/content/motor-control-optimization")
    if not root.exists():
        subprocess.run(["git", "clone", REPO_URL, str(root)], check=True)
    else:
        subprocess.run(["git", "pull", "--ff-only"], cwd=root, check=False)
    ROOT = root
else:
    ROOT = _find_root(Path.cwd())

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplconfig")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp/xdgcache")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["XDG_CACHE_HOME"]).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib
if not _in_colab():
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams["axes.unicode_minus"] = False

try:
    from IPython.display import display
except Exception:
    display = print


def _git(*args):
    try:
        return subprocess.check_output(["git", *args], cwd=ROOT, text=True).strip()
    except Exception:
        return None

ENV = {
    "root": str(ROOT),
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "git_commit": _git("rev-parse", "HEAD"),
    "git_dirty": bool(_git("status", "--short")),
}

print(json.dumps(ENV, indent=2, ensure_ascii=False))

{
  "root": "/Users/seong-uuimac/Documents/Codex/2026-09-06/referenced-chatgpt-conversation-this-is-an/repo",
  "python": "3.13.9",
  "numpy": "2.3.5",
  "pandas": "2.3.3",
  "git_commit": "94987e2b258908c4477adee8d4a1e26da4faf6ae",
  "git_dirty": true
}


In [2]:
SAVE_ARTIFACTS = True
RANDOM_SEED = 42

RESULT_TABLE_DIR = Path("results") / "tables"
RESULT_FIGURE_DIR = Path("results") / "figures"
if SAVE_ARTIFACTS:
    RESULT_TABLE_DIR.mkdir(parents=True, exist_ok=True)
    RESULT_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("SAVE_ARTIFACTS:", SAVE_ARTIFACTS)
print("RANDOM_SEED:", RANDOM_SEED)

SAVE_ARTIFACTS: True
RANDOM_SEED: 42


## Section 1 — Fixed Conditions and v1 Constraints

이 section에서는 docs와 이전 experiment output에 이미 고정된 값을 불러온다.

- Nominal motor parameters: `src.motor.dc_motor.nominal_dc_motor_params()`
- Reference speed, simulation time, dt: `results/tables/reference_selection_record.json`
- Baseline PID gains: `results/tables/baseline_pid_tuning_record.json`
- Cost weights: `src.optimization.cost_function.compute_cost()` default
- v1 constraints: `src.optimization.cost_function.V1_CONSTRAINTS`

Optimizer가 바꾸는 것은 오직 `K_p`, `K_i`, `K_d`이다.

In [3]:
from src.controller.pid import PIDGains
from src.motor.dc_motor import NOMINAL_MOTOR_SOURCE, nominal_dc_motor_params
from src.optimization.cost_function import (
    V1_CONSTRAINTS,
    accepted_baseline,
    compute_cost,
    is_feasible_v1,
    v1_constraint_checks,
)
from src.optimization.pid_optimization import (
    PIDOptimizationConfig,
    constrained_bayesian_optimization_pid,
    constrained_random_search_pid,
    evaluate_pid_candidate,
)
from src.simulation.pid_simulation import reference_from_reachable_speed, simulate_pid

params = nominal_dc_motor_params()
V_MAX = 12.0

REFERENCE_FILE = RESULT_TABLE_DIR / "reference_selection_record.json"
BASELINE_FILE = RESULT_TABLE_DIR / "baseline_pid_tuning_record.json"

if REFERENCE_FILE.exists():
    reference_record = json.loads(REFERENCE_FILE.read_text(encoding="utf-8"))
else:
    reference_record = {
        "V_max": V_MAX,
        "omega_ss_max_rad_s": params.no_load_steady_state_speed(V_MAX),
        "reference_fraction": 0.50,
        "omega_ref_rad_s": round(reference_from_reachable_speed(params, V_MAX, fraction=0.50), 2),
        "simulation_time_s": 10.0,
        "dt_s": 0.001,
        "note": "Recomputed because 01 reference record was not found.",
    }

if BASELINE_FILE.exists():
    baseline_source_record = json.loads(BASELINE_FILE.read_text(encoding="utf-8"))
else:
    baseline_source_record = {
        "K_p_baseline": 0.8,
        "K_i_baseline": 2.0,
        "K_d_baseline": 0.002,
        "note": "Fallback from docs/experiments/baseline_pid_tuning.md because baseline result file was not found.",
    }

OMEGA_REF = float(reference_record["omega_ref_rad_s"])
SIMULATION_TIME = float(reference_record.get("simulation_time_s", 10.0))
DT = float(reference_record.get("dt_s", 0.001))
LOAD_TORQUE = 0.0

COST_WEIGHTS = {
    "w_tracking": 0.60,
    "w_overshoot": 0.25,
    "w_control": 0.15,
}

config = PIDOptimizationConfig(
    omega_ref=OMEGA_REF,
    V_max=V_MAX,
    simulation_time=SIMULATION_TIME,
    dt=DT,
    load_torque=LOAD_TORQUE,
)

fixed_condition_summary = pd.DataFrame([{
    "R": params.R,
    "L": params.L,
    "J_m": params.J_m,
    "b": params.b,
    "K_t": params.K_t,
    "K_e": params.K_e,
    "V_max": V_MAX,
    "omega_ref": OMEGA_REF,
    "simulation_time": SIMULATION_TIME,
    "dt": DT,
    "load_torque": LOAD_TORQUE,
}])

constraint_summary = pd.DataFrame([{
    "overshoot_percent_max": 100.0 * V1_CONSTRAINTS.overshoot_limit,
    "steady_state_error_percent_max": 100.0 * V1_CONSTRAINTS.steady_state_error_limit,
    "settling_time_max_s": V1_CONSTRAINTS.settling_time_limit,
    "saturation_percent_max_exclusive": 100.0 * V1_CONSTRAINTS.saturation_fraction_limit,
    "stable_no_divergence": True,
}])

display(fixed_condition_summary)
display(constraint_summary)

,R,L,J_m,b,K_t,K_e,V_max,omega_ref,simulation_time,dt,load_torque
0,0.18644,0.0063,0.013767,0.049813,0.020375,0.020375,12.0,12.6,10.0,0.001,0.0


,overshoot_percent_max,steady_state_error_percent_max,settling_time_max_s,saturation_percent_max_exclusive,stable_no_divergence
0,10.0,2.0,2.0,5.0,True


## Section 2 — Manual Baseline Benchmark

Baseline gains는 `02_pid_baseline_tuning.ipynb`에서 response-based acceptance rule로 고정한 값이다.

```text
K_p_baseline = 0.80
K_i_baseline = 2.00
K_d_baseline = 0.002
```

이 baseline은 optimizer input이 아니라, optimization 이후 비교할 benchmark이다.

In [4]:
baseline_gains = PIDGains(
    K_p=float(baseline_source_record["K_p_baseline"]),
    K_i=float(baseline_source_record["K_i_baseline"]),
    K_d=float(baseline_source_record["K_d_baseline"]),
)

baseline_result = simulate_pid(
    motor_params=params,
    gains=baseline_gains,
    omega_ref=OMEGA_REF,
    V_max=V_MAX,
    simulation_time=SIMULATION_TIME,
    dt=DT,
    load_torque=LOAD_TORQUE,
)
baseline_cost = compute_cost(baseline_result, omega_ref=OMEGA_REF, V_max=V_MAX)
baseline_constraints = v1_constraint_checks(baseline_cost)

baseline_summary = pd.DataFrame([{
    "Controller": "Manual Baseline",
    "K_p": baseline_gains.K_p,
    "K_i": baseline_gains.K_i,
    "K_d": baseline_gains.K_d,
    "J_total": baseline_cost["total"],
    "J_tracking": baseline_cost["tracking"],
    "J_overshoot": baseline_cost["overshoot_cost"],
    "J_control": baseline_cost["control"],
    "overshoot_percent": baseline_cost["overshoot_percent"],
    "settling_time": baseline_cost["settling_time"],
    "steady_state_error_percent": baseline_cost["steady_state_error_percent"],
    "voltage_max_abs": baseline_cost["voltage_max_abs"],
    "saturation_percent": baseline_cost["saturation_percent"],
    "omega_final": baseline_cost["omega_final"],
    "feasible": is_feasible_v1(baseline_cost),
}])

assert bool(baseline_summary.loc[0, "feasible"]), "Manual baseline must satisfy v1 constraints."
display(baseline_summary.T.rename(columns={0: "value"}))

,value
Controller,Manual Baseline
K_p,0.8
K_i,2.0
K_d,0.002
J_total,0.038177
J_tracking,0.000114
J_overshoot,0.0
J_control,0.254054
overshoot_percent,0.0
settling_time,1.322


## Section 3 — Constrained Optimizer Setup

Search bounds와 evaluation budget은 현재 repo의 v1 design choice로 고정한다.

- Bounds는 baseline 주변을 포함하지만 지나치게 넓지 않은 conservative range이다.
- Random Search와 Bayesian Optimization은 같은 `80`회 evaluation budget을 사용한다.
- 최종 `Constrained Classical`은 두 method의 feasible best 중 `J_total`이 더 낮은 controller로 선택한다.

In [5]:
GAIN_BOUNDS = {
    "K_p": (0.20, 1.50),
    "K_i": (0.50, 4.00),
    "K_d": (0.00, 0.01),
}

N_RANDOM_TRIALS = 80
N_BAYESIAN_TRIALS = 80
BAYESIAN_INITIAL_TRIALS = 8
BAYESIAN_ACQUISITION_POOL_SIZE = 1200

search_summary = pd.DataFrame([
    {"Gain": gain, "Lower Bound": low, "Upper Bound": high}
    for gain, (low, high) in GAIN_BOUNDS.items()
])

budget_summary = pd.DataFrame([
    {"Method": "Constrained Random Search", "Evaluation Budget": N_RANDOM_TRIALS},
    {"Method": "Constrained Bayesian Optimization", "Evaluation Budget": N_BAYESIAN_TRIALS},
])

display(search_summary)
display(budget_summary)

,Gain,Lower Bound,Upper Bound
0,K_p,0.2,1.50
1,K_i,0.5,4.00
2,K_d,0.0,0.01


,Method,Evaluation Budget
0,Constrained Random Search,80
1,Constrained Bayesian Optimization,80


## Section 4 — Run Constrained Searches

각 candidate는 closed-loop simulation 후 v1 constraints를 검사한다.
Infeasible candidate는 optimizer objective에서 큰 penalty를 받으며, 최종 selection에서는 hard reject된다.

In [6]:
random_records, random_best = constrained_random_search_pid(
    motor_params=params,
    bounds=GAIN_BOUNDS,
    n_trials=N_RANDOM_TRIALS,
    config=config,
    seed=RANDOM_SEED,
)

bayesian_records, bayesian_best = constrained_bayesian_optimization_pid(
    motor_params=params,
    bounds=GAIN_BOUNDS,
    n_trials=N_BAYESIAN_TRIALS,
    config=config,
    seed=RANDOM_SEED,
    n_initial=BAYESIAN_INITIAL_TRIALS,
    acquisition_pool_size=BAYESIAN_ACQUISITION_POOL_SIZE,
)

random_history = pd.DataFrame(random_records)
bayesian_history = pd.DataFrame(bayesian_records)


def feasible_best_so_far(history):
    best = math.inf
    values = []
    for _, row in history.iterrows():
        if bool(row["feasible"]):
            best = min(best, float(row["total"]))
        values.append(np.nan if not np.isfinite(best) else best)
    return values

random_history["feasible_best_so_far"] = feasible_best_so_far(random_history)
bayesian_history["feasible_best_so_far"] = feasible_best_so_far(bayesian_history)

method_selection = pd.DataFrame([
    {
        "Method": "Constrained Random Search",
        "feasible_candidates": int(random_history["feasible"].sum()),
        "best_J_total": random_best["total"],
        "K_p": random_best["K_p"],
        "K_i": random_best["K_i"],
        "K_d": random_best["K_d"],
        "settling_time": random_best["settling_time"],
        "steady_state_error_percent": random_best["steady_state_error_percent"],
    },
    {
        "Method": "Constrained Bayesian Optimization",
        "feasible_candidates": int(bayesian_history["feasible"].sum()),
        "best_J_total": bayesian_best["total"],
        "K_p": bayesian_best["K_p"],
        "K_i": bayesian_best["K_i"],
        "K_d": bayesian_best["K_d"],
        "settling_time": bayesian_best["settling_time"],
        "steady_state_error_percent": bayesian_best["steady_state_error_percent"],
    },
])

display(method_selection)

,Method,feasible_candidates,best_J_total,K_p,K_i,K_d,settling_time,steady_state_error_percent
0,Constrained Random Search,45,0.036768,0.256945,1.040013,0.006830,1.550,4.536054e-10
1,Constrained Bayesian Optimization,51,0.036702,0.215097,0.996516,0.000855,1.547,4.709601e-10


## Section 5 — Best Feasible Constrained PID

두 constrained method의 feasible best 후보 중 total cost가 더 낮은 controller를 최종 `Constrained Classical` PID로 선택한다.

In [7]:
candidate_bests = [
    ("Constrained Random Search", random_best),
    ("Constrained Bayesian Optimization", bayesian_best),
]
selected_method, selected_best = min(candidate_bests, key=lambda item: item[1]["total"])
constrained_classical_gains = PIDGains(
    K_p=float(selected_best["K_p"]),
    K_i=float(selected_best["K_i"]),
    K_d=float(selected_best["K_d"]),
)

assert bool(selected_best["feasible"]), "Selected constrained PID must be feasible."
print("Selected Constrained Classical method:", selected_method)
display(pd.DataFrame([selected_best]).T.rename(columns={0: "value"}))

Selected Constrained Classical method: Constrained Bayesian Optimization


,value
K_p,0.215097
K_i,0.996516
K_d,0.000855
total,0.036702
tracking,0.000611
overshoot_cost,0.0
control,0.242238
omega_final,12.6
omega_max,12.601091
omega_min,0.000316


## Section 6 — Manual Baseline vs Constrained Classical Comparison Table

최종 비교표에는 두 controller만 표시한다.

In [8]:
def simulate_and_summarize(label, gains):
    result = simulate_pid(
        motor_params=params,
        gains=gains,
        omega_ref=OMEGA_REF,
        V_max=V_MAX,
        simulation_time=SIMULATION_TIME,
        dt=DT,
        load_torque=LOAD_TORQUE,
    )
    cost = compute_cost(result, omega_ref=OMEGA_REF, V_max=V_MAX)
    return result, {
        "Controller": label,
        "K_p": gains.K_p,
        "K_i": gains.K_i,
        "K_d": gains.K_d,
        "J_total": cost["total"],
        "J_tracking": cost["tracking"],
        "J_overshoot": cost["overshoot_cost"],
        "J_control": cost["control"],
        "overshoot_percent": cost["overshoot_percent"],
        "settling_time": cost["settling_time"],
        "steady_state_error_percent": cost["steady_state_error_percent"],
        "voltage_max_abs": cost["voltage_max_abs"],
        "saturation_percent": cost["saturation_percent"],
        "omega_final": cost["omega_final"],
        "feasible": is_feasible_v1(cost),
    }

responses = {}
rows = []
for label, gains in [
    ("Manual Baseline", baseline_gains),
    ("Constrained Classical", constrained_classical_gains),
]:
    result, row = simulate_and_summarize(label, gains)
    responses[label] = result
    rows.append(row)

comparison = pd.DataFrame(rows)
baseline_j = float(comparison.loc[comparison["Controller"] == "Manual Baseline", "J_total"].iloc[0])
comparison["J_improvement_vs_baseline_percent"] = (
    (baseline_j - comparison["J_total"]) / baseline_j * 100.0
)

comparison_columns = [
    "Controller",
    "K_p", "K_i", "K_d",
    "J_total", "J_tracking", "J_overshoot", "J_control",
    "overshoot_percent", "settling_time", "steady_state_error_percent",
    "voltage_max_abs", "saturation_percent", "omega_final",
    "feasible", "J_improvement_vs_baseline_percent",
]

assert comparison["feasible"].all(), "Both final comparison controllers must satisfy v1 constraints."
display(comparison[comparison_columns])

,Controller,K_p,K_i,K_d,J_total,J_tracking,J_overshoot,J_control,overshoot_percent,settling_time,steady_state_error_percent,voltage_max_abs,saturation_percent,omega_final,feasible,J_improvement_vs_baseline_percent
0,Manual Baseline,0.800000,2.000000,0.002000,0.038177,0.000114,0.000000e+00,0.254054,0.00000,1.322,2.158821e-07,10.242803,0.0,12.6,True,0.000000
1,Constrained Classical,0.215097,0.996516,0.000855,0.036702,0.000611,7.499755e-09,0.242238,0.00866,1.547,4.709601e-10,6.001635,0.0,12.6,True,3.861902


## Section 7 — Speed Response Comparison Plot

Reference line을 함께 표시해서 response speed와 settling behavior를 확인한다.

In [9]:
fig, ax = plt.subplots(figsize=(10, 4.8))

for label, result in responses.items():
    ax.plot(result["time"], result["omega"], label=label)
ax.axhline(OMEGA_REF, linestyle="--", color="black", linewidth=1, label="Reference")
ax.set_xlabel("Time [s]")
ax.set_ylabel("omega [rad/s]")
ax.set_title("Speed Response: Manual Baseline vs Constrained Classical")
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
if SAVE_ARTIFACTS:
    plt.savefig(RESULT_FIGURE_DIR / "constrained_pid_speed_response.png", dpi=160)
plt.show()

/var/folders/js/s5xbjpb56mn1ysk430xc81w40000gn/T/ipykernel_45060/1281611209.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 8 — Control Voltage Comparison Plot

Voltage limit line을 함께 표시해서 control effort와 saturation 여부를 확인한다.

In [10]:
fig, ax = plt.subplots(figsize=(10, 4.8))

for label, result in responses.items():
    ax.plot(result["time"], result["voltage"], label=label)
ax.axhline(V_MAX, linestyle="--", color="tab:red", linewidth=1, label="+Vmax")
ax.axhline(-V_MAX, linestyle="--", color="tab:red", linewidth=1, label="-Vmax")
ax.set_xlabel("Time [s]")
ax.set_ylabel("voltage [V]")
ax.set_title("Control Voltage: Manual Baseline vs Constrained Classical")
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
if SAVE_ARTIFACTS:
    plt.savefig(RESULT_FIGURE_DIR / "constrained_pid_control_voltage.png", dpi=160)
plt.show()

/var/folders/js/s5xbjpb56mn1ysk430xc81w40000gn/T/ipykernel_45060/2878794042.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 9 — Constraint Satisfaction Table / Analysis

Baseline과 constrained classical controller가 같은 v1 constraints를 만족하는지 확인한다.

In [11]:
constraint_rows = []
for label, result in responses.items():
    cost = compute_cost(result, omega_ref=OMEGA_REF, V_max=V_MAX)
    checks = v1_constraint_checks(cost)
    constraint_rows.append({
        "Controller": label,
        "stable": checks["stable"],
        "overshoot <= 10%": checks["overshoot"],
        "SSE <= 2%": checks["steady_state_error"],
        "settling_time <= 2.0s": checks["settling_time"],
        "saturation < 5%": checks["saturation"],
        "feasible": is_feasible_v1(cost),
    })

constraint_table = pd.DataFrame(constraint_rows)
assert constraint_table["feasible"].all(), "All final controllers must satisfy every v1 constraint."
display(constraint_table)

,Controller,stable,overshoot <= 10%,SSE <= 2%,settling_time <= 2.0s,saturation < 5%,feasible
0,Manual Baseline,True,True,True,True,True,True
1,Constrained Classical,True,True,True,True,True,True


## Section 10 — Method Selection Evidence

Cost history는 최종 비교 그래프가 아니라, constrained classical method 선택 근거로 저장한다.

In [12]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(random_history["trial"], random_history["feasible_best_so_far"], marker="o", label="Constrained Random Search")
ax.plot(bayesian_history["trial"], bayesian_history["feasible_best_so_far"], marker="o", label="Constrained Bayesian Optimization")
ax.axhline(baseline_j, linestyle="--", color="black", linewidth=1, label="Manual Baseline J")
ax.set_xlabel("Evaluation trial")
ax.set_ylabel("Best feasible J so far")
ax.set_title("Constrained Classical Method Selection Evidence")
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
if SAVE_ARTIFACTS:
    plt.savefig(RESULT_FIGURE_DIR / "constrained_pid_cost_history.png", dpi=160)
plt.show()

/var/folders/js/s5xbjpb56mn1ysk430xc81w40000gn/T/ipykernel_45060/985525691.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 11 — Save Artifacts

이 notebook은 constrained optimization 결과를 `results/tables/`와 `results/figures/`에 저장한다.

In [13]:
def _json_safe(value):
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (np.floating, float)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, (np.integer, int)):
        return int(value)
    if isinstance(value, dict):
        return {k: _json_safe(v) for k, v in value.items()}
    if isinstance(value, list):
        return [_json_safe(v) for v in value]
    return value

try:
    import sklearn
    sklearn_version = sklearn.__version__
except Exception:
    sklearn_version = None
try:
    import scipy
    scipy_version = scipy.__version__
except Exception:
    scipy_version = None

RUN_CONFIG = {
    "random_seed": RANDOM_SEED,
    "gain_bounds": GAIN_BOUNDS,
    "gain_bounds_note": "v1 conservative design choice around the manual baseline gains",
    "evaluation_budget": {
        "Constrained Random Search": N_RANDOM_TRIALS,
        "Constrained Bayesian Optimization": N_BAYESIAN_TRIALS,
    },
    "bayesian_initial_trials": BAYESIAN_INITIAL_TRIALS,
    "bayesian_acquisition_pool_size": BAYESIAN_ACQUISITION_POOL_SIZE,
    "fixed_conditions": fixed_condition_summary.iloc[0].to_dict(),
    "cost_weights": COST_WEIGHTS,
    "v1_constraints": constraint_summary.iloc[0].to_dict(),
    "environment": {
        **ENV,
        "matplotlib": matplotlib.__version__,
        "sklearn": sklearn_version,
        "scipy": scipy_version,
    },
}

summary_payload = {
    "selected_method": selected_method,
    "comparison": comparison[comparison_columns].to_dict(orient="records"),
    "constraint_satisfaction": constraint_table.to_dict(orient="records"),
    "method_selection": method_selection.to_dict(orient="records"),
    "random_search_best": dict(random_best),
    "bayesian_optimization_best": dict(bayesian_best),
    "run_config": RUN_CONFIG,
    "motor_source": NOMINAL_MOTOR_SOURCE,
}

if SAVE_ARTIFACTS:
    comparison[comparison_columns].to_csv(RESULT_TABLE_DIR / "constrained_pid_optimization_summary.csv", index=False)
    method_selection.to_csv(RESULT_TABLE_DIR / "constrained_pid_method_selection.csv", index=False)
    constraint_table.to_csv(RESULT_TABLE_DIR / "constrained_pid_constraint_satisfaction.csv", index=False)
    random_history.to_csv(RESULT_TABLE_DIR / "constrained_pid_random_search_history.csv", index=False)
    bayesian_history.to_csv(RESULT_TABLE_DIR / "constrained_pid_bayesian_optimization_history.csv", index=False)
    with open(RESULT_TABLE_DIR / "constrained_pid_optimization_summary.json", "w", encoding="utf-8") as f:
        json.dump(_json_safe(summary_payload), f, indent=2, ensure_ascii=False)

print("Constrained PID optimization records saved." if SAVE_ARTIFACTS else "SAVE_ARTIFACTS=False, no files saved.")
print(json.dumps(_json_safe({
    "selected_method": selected_method,
    "baseline_J": baseline_j,
    "constrained_classical_J": float(comparison.loc[comparison["Controller"] == "Constrained Classical", "J_total"].iloc[0]),
    "J_improvement_vs_baseline_percent": float(comparison.loc[comparison["Controller"] == "Constrained Classical", "J_improvement_vs_baseline_percent"].iloc[0]),
}), indent=2, ensure_ascii=False))

Constrained PID optimization records saved.
{
  "selected_method": "Constrained Bayesian Optimization",
  "baseline_J": 0.03817651842324221,
  "constrained_classical_J": 0.03670217885813229,
  "J_improvement_vs_baseline_percent": 3.861901571968205
}


## Final Summary

이 notebook은 `Manual Baseline`과 `Constrained Classical`을 같은 plant와 같은 simulation condition에서 비교한다.

Constrained Random Search와 Constrained Bayesian Optimization은 같은 budget으로 실행하며,
feasible best J가 더 낮은 method를 최종 `Constrained Classical` PID로 선택한다.